# NetraEdge — Model Training Pipeline

**Offline Face Recognition + Liveness Detection**

This notebook trains both models end-to-end and exports them as ONNX + TFLite (INT8).

**Steps:**
1. Click **Runtime → Run All** (or Ctrl+Shift+Enter on each cell)
2. Wait for training to complete (~5 min on GPU)
3. Models are saved to `models/` directory
4. Copy `.tflite` files to `packages/app/android/app/src/main/assets/`

**Runtime:** GPU T4 recommended (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Cell 1: Setup
!pip install -q onnx onnxscript tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import time
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')\n
# Create output directories
os.makedirs('models', exist_ok=True)
os.makedirs('models/tflite', exist_ok=True)
print('Setup complete!')

In [ ]:
# Cell 2: MobileFaceNet Architecture

class DWSep(nn.Module):
    """Depthwise separable convolution."""
    def __init__(self, ic, oc, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, stride, 1, groups=ic, bias=False)
        self.bn1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(oc)

    def forward(self, x):
        return F.relu(self.bn2(self.pw(F.relu(self.bn1(self.dw(x)), inplace=True))), inplace=True)


class SE(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 8)
        self.fc1 = nn.Linear(ch, mid)
        self.fc2 = nn.Linear(mid, ch)

    def forward(self, x):
        b, c, _, _ = x.size()
        s = F.adaptive_avg_pool2d(x, 1).view(b, c)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s)).view(b, c, 1, 1)
        return x * s


class MB(nn.Module):
    """MobileBlock with residual connection."""
    def __init__(self, ic, oc, stride=1):
        super().__init__()
        self.expand = nn.Sequential(
            nn.Conv2d(ic, ic * 2, 1, bias=False),
            nn.BatchNorm2d(ic * 2),
            nn.ReLU(inplace=True),
        )
        self.dwsep = DWSep(ic * 2, oc, stride)
        self.se = SE(oc)
        self.residual = (stride == 1 and ic == oc)

    def forward(self, x):
        out = self.expand(x)
        out = self.dwsep(out)
        out = self.se(out)
        if self.residual:
            out = out + x
        return out


class MobileFaceNet(nn.Module):
    """
    MobileFaceNet for face recognition.
    Input: 112x112x3 RGB image
    Output: 128-d L2-normalized embedding
    """
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            MB(64, 64, stride=2),
            MB(64, 64),
            MB(64, 128, stride=2),
            MB(128, 128),
            MB(128, 128),
            MB(128, 256, stride=2),
            MB(256, 256),
            MB(256, 256),
            MB(256, 512, stride=2),
            MB(512, 512),
        )
        self.final_dw = nn.Sequential(
            nn.Conv2d(512, 512, 3, 1, 1, groups=512, bias=False),
            nn.BatchNorm2d(512),
        )
        self.fc = nn.Linear(512, embedding_dim)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = F.relu(self.final_dw(x), inplace=True)
        x = F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x


model = MobileFaceNet(128).to(device)
params = sum(p.numel() for p in model.parameters())
size_mb = params * 4 / 1024 / 1024
print(f'MobileFaceNet: {params:,} params, ~{size_mb:.1f} MB (FP32)')
print(f'INT8 estimate: ~{size_mb/4:.1f} MB')

In [ ]:
# Cell 3: Synthetic Training Data

class FaceDataset(torch.utils.data.Dataset):
    """Synthetic face dataset for training demonstration."""
    def __init__(self, n_identities=50, imgs_per_id=20, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        np.random.seed(42)
        for identity in range(n_identities):
            base = np.random.rand(3, 112, 112).astype(np.float32)
            for j in range(imgs_per_id):
                img = base + np.random.randn(3, 112, 112).astype(np.float32) * 0.15
                img = np.clip(img, 0, 1)
                # Random horizontal flip
                if np.random.rand() > 0.5:
                    img = img[:, :, ::-1].copy()
                # Random brightness
                img = img * (0.8 + np.random.rand() * 0.4)
                img = np.clip(img, 0, 1)
                self.images.append(img)
                self.labels.append(identity)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


train_ds = FaceDataset(n_identities=50, imgs_per_id=20)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True
)
print(f'Training data: {len(train_ds)} samples, {50} identities')
print(f'Batches per epoch: {len(train_loader)}')

In [ ]:
# Cell 4: Train Recognition Model

def train_recognition(epochs=10):
    model = MobileFaceNet(128).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    print(f'\nTraining MobileFaceNet for {epochs} epochs...')
    print('-' * 50)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        t0 = time.time()

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            embeddings = model(images)
            # Use cosine similarity as logits for classification
            logits = embeddings @ embeddings.t() * 64
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        acc = 100.0 * correct / total
        elapsed = time.time() - t0

        if acc > best_acc:
            best_acc = acc
            torch.save({'model_state_dict': model.state_dict()}, 'models/rec_best.pt')

        print(f'  Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc:5.1f}% | Time: {elapsed:.1f}s')

    print(f'\nBest accuracy: {best_acc:.1f}%')
    print(f'Saved: models/rec_best.pt')
    return model


rec_model = train_recognition(epochs=10)

In [ ]:
# Cell 5: Export Recognition Model

def export_to_onnx(model, output_path, input_size=(1, 3, 112, 112)):
    model.eval()
    dummy = torch.randn(*input_size).to(device)
    torch.onnx.export(
        model, dummy, output_path,
        opset_version=13,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    )
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f'  Exported: {output_path} ({size_mb:.2f} MB)')
    return size_mb


print('Exporting recognition model...')
rec_size = export_to_onnx(rec_model, 'models/face_recognition.onnx')
print('Done!')

In [ ]:
# Cell 6: LivenessCNN Architecture + Dataset

class LivBlock(nn.Module):
    """Depthwise separable block for liveness."""
    def __init__(self, ic, oc, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, stride, 1, groups=ic, bias=False)
        self.bn1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(oc)

    def forward(self, x):
        return F.relu(self.bn2(self.pw(F.relu(self.bn1(self.dw(x)), True))), True)


class LivenessCNN(nn.Module):
    """
    Liveness detection CNN.
    Input: 112x112x3 RGB image
    Output: 3-class logits [real, print, screen]
    """
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            LivBlock(32, 64),
            LivBlock(64, 128, stride=2),
            LivBlock(128, 256, stride=2),
            LivBlock(256, 256, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.15),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class LivDataset(torch.utils.data.Dataset):
    """Synthetic liveness dataset: real=0, print=1, screen=2."""
    def __init__(self, n_samples=3000):
        self.images = []
        self.labels = []
        np.random.seed(42)
        for i in range(n_samples):
            label = i % 3
            img = np.random.rand(3, 112, 112).astype(np.float32)
            if label == 0:  # Real skin texture
                img = img * 0.3 + 0.4
            elif label == 1:  # Printed photo
                img = img * 0.1 + 0.7
            else:  # Screen display
                img = img * 0.5 + 0.2
                # Add screen-like horizontal lines
                img[:, ::4, :] *= 0.8
            self.images.append(np.clip(img, 0, 1))
            self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return torch.from_numpy(self.images[idx]), self.labels[idx]


liv_train_ds = LivDataset(3000)
liv_train_loader = torch.utils.data.DataLoader(
    liv_train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True
)

liv_model_check = LivenessCNN(3).to(device)
params = sum(p.numel() for p in liv_model_check.parameters())
print(f'LivenessCNN: {params:,} params, ~{params*4/1024/1024:.1f} MB (FP32)')
print(f'INT8 estimate: ~{params*4/1024/1024/4:.1f} MB')
print(f'Training data: {len(liv_train_ds)} samples, 3 classes')

In [ ]:
# Cell 7: Train Liveness Model

def train_liveness(epochs=10):
    model = LivenessCNN(3).to(device)
    # Higher weight for spoof classes (print=1.5, screen=1.5)
    weight = torch.tensor([1.0, 1.5, 1.5]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    print(f'\nTraining LivenessCNN for {epochs} epochs...')
    print('-' * 50)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        t0 = time.time()

        for images, labels in liv_train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        avg_loss = total_loss / len(liv_train_loader)
        acc = 100.0 * correct / total
        elapsed = time.time() - t0

        if acc > best_acc:
            best_acc = acc
            torch.save({'model_state_dict': model.state_dict()}, 'models/liv_best.pt')

        print(f'  Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc:5.1f}% | Time: {elapsed:.1f}s')

    print(f'\nBest accuracy: {best_acc:.1f}%')
    print(f'Saved: models/liv_best.pt')
    return model


liv_model = train_liveness(epochs=10)

In [ ]:
# Cell 8: Export Liveness Model

print('Exporting liveness model...')
liv_size = export_to_onnx(liv_model, 'models/liveness_detector.onnx')
print('Done!')

In [ ]:
# Cell 9: Convert to TFLite (INT8 Quantized)

def convert_to_tflite(onnx_path, tflite_path):
    """Convert ONNX to TFLite with INT8 quantization."""
    try:
        import onnx
        from onnx import TensorProto
        import struct

        # Load ONNX model to get input shape
        onnx_model = onnx.load(onnx_path)
        print(f'  Loaded: {onnx_path}')

        # For VS Code / local: use onnx2tf if available
        try:
            import onnx2tf
            onnx2tf.convert(
                input_onnx_file_path=onnx_path,
                output_folder_path=os.path.dirname(tflite_path),
                non_verbose=True,
            )
            # Find the generated tflite file
            for f in os.listdir(os.path.dirname(tflite_path)):
                if f.endswith('.tflite'):
                    src = os.path.join(os.path.dirname(tflite_path), f)
                    if src != tflite_path:
                        os.replace(src, tflite_path)
            print(f'  Converted via onnx2tf: {tflite_path}')
        except ImportError:
            # Fallback: use tf.lite if available
            try:
                import tensorflow as tf
                # Try TFLiteConverter from saved model
                converter = tf.lite.TFLiteConverter.from_concrete_functions([])
            except Exception:
                print(f'  TFLite conversion skipped (install onnx2tf: pip install onnx2tf)')
                print(f'  Using ONNX as primary model format')
                return False
    except Exception as e:
        print(f'  TFLite conversion error: {e}')
        return False
    return True


print('Converting to TFLite...')
rec_ok = convert_to_tflite('models/face_recognition.onnx', 'models/face_recognition.tflite')
liv_ok = convert_to_tflite('models/liveness_detector.onnx', 'models/liveness_detector.tflite')

# INT8 quantization (if TFLite files exist)
if os.path.exists('models/face_recognition.tflite'):
    try:
        import tensorflow as tf
        for name in ['face_recognition', 'liveness_detector']:
            path = f'models/{name}.tflite'
            if os.path.exists(path):
                converter = tf.lite.TFLiteConverter.from_saved_model(path.replace('.tflite', ''))
                converter.optimizations = [tf.lite.Optimize.DEFAULT]
                tflite_quant = converter.convert()
                with open(path.replace('.tflite', '_int8.tflite'), 'wb') as f:
                    f.write(tflite_quant)
        print('  INT8 quantization complete')
    except Exception:
        print('  INT8 quantization skipped (TensorFlow not available)')

print('\nTFLite conversion done!')

In [ ]:
# Cell 10: Summary + Copy Models

import shutil

print('=' * 50)
print('  NetraEdge Training Complete!')
print('=' * 50)
print()

# List all generated files
print('Generated models:')
for f in sorted(Path('models').rglob('*')):
    if f.is_file() and f.name != 'README.md':
        size = f.stat().st_size / 1024 / 1024
        print(f'  {f} ({size:.2f} MB)')

# Try to copy to Android assets
assets_dir = 'packages/app/android/app/src/main/assets'
if os.path.exists(os.path.dirname(assets_dir)):
    os.makedirs(assets_dir, exist_ok=True)
    for name in ['face_recognition.tflite', 'liveness_detector.tflite']:
        src = f'models/{name}'
        if os.path.exists(src):
            shutil.copy2(src, f'{assets_dir}/{name}')
            print(f'  Copied: {src} -> {assets_dir}/{name}')
    # Also try _int8 variants
    for name in ['face_recognition_int8.tflite', 'liveness_detector_int8.tflite']:
        src = f'models/{name}'
        if os.path.exists(src):
            out_name = name.replace('_int8', '')
            shutil.copy2(src, f'{assets_dir}/{out_name}')
            print(f'  Copied: {src} -> {assets_dir}/{out_name}')
else:
    print(f'\n  Android assets dir not found. Copy manually:')
    print(f'    models/*.tflite -> packages/app/android/app/src/main/assets/')

print()
print('Next steps:')
print('  1. Copy .tflite files to packages/app/android/app/src/main/assets/')
print('  2. Run: cd packages/app && npx react-native run-android')
print('  3. Test: Enroll a face -> Verify identity')
print()
print('Models work offline. No internet required.')